# MLflow Integration in noted

This notebook demonstrates how **noted** integrates with MLflow for experiment tracking. The MLflow tracking server and experiment name are automatically configured when a kernel starts — no setup code needed.

## 1. Verify the Connection

The kernel automatically sets `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME` environment variables. The snippet below confirms these are injected and that `mlflow` can reach the tracking server.

In [1]:
import os
import mlflow

print("Tracking URI: ", mlflow.get_tracking_uri())
print("Experiment  : ", os.environ.get('MLFLOW_EXPERIMENT_NAME', '(not set)'))

# Confirm server is reachable
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(os.environ.get('MLFLOW_EXPERIMENT_NAME'))
if experiment:
    print(f"Experiment id={experiment.experiment_id}  lifecycle={experiment.lifecycle_stage}")
else:
    print("Experiment not found yet — it will be created on first run.")

Tracking URI:  http://mlflow:5000
Experiment  :  Examples
Experiment id=2  lifecycle=active


## 2. Manual Parameter & Metric Logging

Use `mlflow.start_run()` as a context manager. Inside the block you can log:
- **params** — hyperparameters and fixed settings (strings/numbers)
- **metrics** — scalar values that may change over steps
- **tags** — arbitrary string key-value annotations

In [2]:
import mlflow

with mlflow.start_run(run_name="manual-example") as run:
    # Log hyperparameters
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("epochs", 10)

    # Simulate training — log a metric at each epoch
    for epoch in range(1, 11):
        loss = 1.0 / (epoch + 1)
        accuracy = 1 - loss
        mlflow.log_metric("loss", loss, step=epoch)
        mlflow.log_metric("accuracy", accuracy, step=epoch)

    # Add a tag
    mlflow.set_tag("author", "notebook")

    print(f"Run id  : {run.info.run_id}")
    print(f"Run name: {run.info.run_name}")

Run id  : d65fb1db85044dafa2b305d524ac4b09
Run name: manual-example
🏃 View run manual-example at: http://mlflow:5000/#/experiments/2/runs/d65fb1db85044dafa2b305d524ac4b09
🧪 View experiment at: http://mlflow:5000/#/experiments/2


## 3. Scikit-learn Autologging

MLflow can automatically capture parameters, metrics, and the trained model from popular ML frameworks. Call `mlflow.sklearn.autolog()` once before training — no explicit `log_param` calls required.

In [3]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.sklearn.autolog()  # capture everything automatically

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="sklearn-autolog"):
    clf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Test accuracy: {acc:.4f}")
    print("Parameters, metrics, and model logged automatically.")

2026/03/11 19:47:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Test accuracy: 1.0000
Parameters, metrics, and model logged automatically.
🏃 View run sklearn-autolog at: http://mlflow:5000/#/experiments/2/runs/c9b9252aa7524bb88d0a89f9893ab065
🧪 View experiment at: http://mlflow:5000/#/experiments/2


## 4. Logging Artifacts

Artifacts are files associated with a run: plots, data snapshots, config files, etc. Log them with `mlflow.log_figure()` (for matplotlib figures) or `mlflow.log_artifact()` (for any file path).

In [4]:
import mlflow
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

with mlflow.start_run(run_name="artifact-example"):
    # Create a simple plot
    epochs = np.arange(1, 21)
    loss = 1 / epochs
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(epochs, loss, marker='o', markersize=4)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Curve")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    # Log the figure directly (no temp file needed)
    mlflow.log_figure(fig, "loss_curve.png")
    plt.close(fig)

    # Log a text artifact
    config = "model: RandomForest\nn_estimators: 100\nmax_depth: 3\n"
    with open("/tmp/config.yaml", "w") as f:
        f.write(config)
    mlflow.log_artifact("/tmp/config.yaml")

    print("Artifacts logged: loss_curve.png, config.yaml")

Artifacts logged: loss_curve.png, config.yaml
🏃 View run artifact-example at: http://mlflow:5000/#/experiments/2/runs/bf28e3eb520142b5ba0615abfaf7add7
🧪 View experiment at: http://mlflow:5000/#/experiments/2


## 5. Querying Runs Programmatically

`mlflow.search_runs()` returns a pandas DataFrame with all runs in the experiment. Use it to compare experiments, find the best run, or build automated reporting.

In [5]:
import mlflow
import os

client = mlflow.tracking.MlflowClient()
experiment_name = os.environ.get('MLFLOW_EXPERIMENT_NAME')
experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    print("No experiment found. Run the cells above first.")
else:
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"]
    )
    print(f"Total runs in '{experiment_name}': {len(runs_df)}")
    cols = [c for c in runs_df.columns if not c.startswith('tags.mlflow')]
    display_cols = ['run_id', 'status', 'start_time'] + \
                   [c for c in cols if c.startswith('params.') or c.startswith('metrics.')]
    print(runs_df[display_cols].head(10).to_string(index=False))

Total runs in 'Examples': 9
                          run_id   status                       start_time  metrics.training_accuracy_score  metrics.training_score  metrics.training_recall_score  metrics.training_precision_score  metrics.training_f1_score  metrics.training_log_loss  metrics.training_roc_auc  metrics.loss  metrics.accuracy params.warm_start params.n_estimators params.max_features params.class_weight params.ccp_alpha params.min_weight_fraction_leaf params.verbose params.max_depth params.monotonic_cst params.min_samples_split params.oob_score params.criterion params.max_samples params.n_jobs params.min_samples_leaf params.min_impurity_decrease params.max_leaf_nodes params.bootstrap params.random_state params.batch_size params.epochs params.learning_rate
bf28e3eb520142b5ba0615abfaf7add7 FINISHED 2026-03-11 19:47:10.099000+00:00                              NaN                     NaN                            NaN                               NaN                        NaN   

## 6. Finding the Best Run and Loading Its Model

Once you have multiple runs you can filter and rank them, then load the best model directly from the tracking server for inference or further analysis.

In [6]:
import mlflow
import os

client = mlflow.tracking.MlflowClient()
experiment_name = os.environ.get('MLFLOW_EXPERIMENT_NAME')
experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    print("Run the cells above first.")
else:
    # Find the run with the best training_score (logged by sklearn autolog)
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="metrics.training_accuracy_score > 0",
        order_by=["metrics.training_accuracy_score DESC"]
    )
    if runs_df.empty:
        print("No autolog runs found. Execute the sklearn autolog cell first.")
    else:
        best = runs_df.iloc[0]
        print(f"Best run   : {best['run_id']}")
        print(f"Train acc  : {best.get('metrics.training_accuracy_score', 'n/a'):.4f}")
        # Load the model logged by autolog
        model_uri = f"runs:/{best['run_id']}/model"
        model = mlflow.sklearn.load_model(model_uri)
        print(f"Model type : {type(model).__name__}")
        print(f"n_estimators: {model.n_estimators}")

Best run   : c9b9252aa7524bb88d0a89f9893ab065
Train acc  : 0.9583


## 7. Accessing the MLflow UI

The MLflow web UI is available through the **noted** icon bar. Click the **MLflow** service icon (flask) in the left sidebar to open the tracking UI in a panel.

From the UI you can:
- Browse all experiments and runs
- Compare metric charts side-by-side
- Download logged artifacts
- Promote model versions to the Model Registry

You can also get a direct link to the current experiment: